# التنبؤ الاستكشافي بالأداء الدراسي

هذا الدفتر يوسّع التحليل الاستكشافي الموجود في النسخة الأصلية إلى مسار تعلم آلة كامل: فحص البيانات، مقارنة نماذج، تقييم على مجموعة اختبار منفصلة، ومراجعة وصفية للأداء حسب النوع.

**الاستخدام المقصود:** مؤشر أولي لبدء متابعة بشرية داعمة. لا يُستخدم لاتخاذ قرار آلي بشأن الطالب.

## 1. تحميل البيانات والأدوات

شغّل الدفتر من مجلد المشروع حتى يجد ملف البيانات وملفات التدريب.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "data" / "xAPI-Edu-Data.csv").exists() and (ROOT / "student-performance-ml" / "data" / "xAPI-Edu-Data.csv").exists():
    ROOT = ROOT / "student-performance-ml"
DATA_PATH = ROOT / "data" / "xAPI-Edu-Data.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError("شغّل الدفتر من مجلد student-performance-ml أو ضع ملف البيانات داخل data/")

from model import FEATURE_COLUMNS, normalize_frame

df = normalize_frame(pd.read_csv(DATA_PATH))
sns.set_theme(style="whitegrid")
print(f"Shape: {df.shape}")
df.head()

## 2. فحص سريع للبيانات

In [ ]:
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing values"))
display(df["Class"].value_counts().rename(index={"L": "Low", "M": "Medium", "H": "High"}).to_frame("students"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x="Class", order=["L", "M", "H"], ax=ax)
ax.set(title="Distribution of performance classes", xlabel="Class (L/M/H)", ylabel="Students")
plt.tight_layout()

## 3. المشاركة والغياب حسب الفئة

المخططات وصفية؛ لا تثبت أن المشاركة أو الغياب يسببان مستوى أداء معينًا.

In [ ]:
engagement = ["raisedhands", "VisITedResources", "AnnouncementsView", "Discussion"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for feature, ax in zip(engagement, axes.flat):
    sns.boxplot(data=df, x="Class", y=feature, order=["L", "M", "H"], ax=ax)
    ax.set_title(feature)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=df, x="StudentAbsenceDays", hue="Class", hue_order=["L", "M", "H"], ax=ax)
ax.set(title="Absence category by performance class", xlabel="Absence days", ylabel="Students")
plt.tight_layout()

In [ ]:
numeric = df.select_dtypes(include="number")
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(numeric.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Numeric-feature correlations")
plt.tight_layout()

## 4. تدريب النماذج ومقارنة الأداء

يقارن التدريب أربعة نماذج باستخدام تحقق متقاطع طبقي متكرر، ثم يقيس النموذج المختار على مجموعة اختبار بقيت خارج عملية الاختيار.

In [ ]:
from train import train_and_evaluate

results = train_and_evaluate(DATA_PATH, ROOT / "artifacts")
display(results["leaderboard"].style.format(precision=3))
display(pd.Series(results["metrics"], name="value").to_frame())

## 5. قراءة النتائج ومراجعة الحدود

In [ ]:
display(results["classification_report"].style.format(precision=3))
display(Image(filename=str(ROOT / "artifacts" / "confusion_matrix.png")))

In [ ]:
display(results["feature_importance"].head(12).style.format(precision=3))
results["feature_importance"].head(12).sort_values("macro_f1_importance_mean").plot.barh(
    x="feature", y="macro_f1_importance_mean", legend=False, figsize=(8, 5),
    title="Permutation importance on the holdout set"
)
plt.xlabel("Change in macro F1 when feature is shuffled")
plt.tight_layout()

In [ ]:
display(results["gender_audit"])
print("Model metrics:")
print(json.dumps(results["metrics"], indent=2, ensure_ascii=False))

## 6. تجربة تقدير لعينة واحدة

هذا المثال يستخدم صفًا من البيانات العامة لتوضيح الواجهة البرمجية. درجة الفئة خضعت لمعايرة داخلية بطريقة sigmoid، لكنها تظل تقديرًا غير يقيني.

In [ ]:
sample = df[FEATURE_COLUMNS].iloc[[0]]
predicted_class = results["model"].predict(sample)[0]
scores = results["model"].predict_proba(sample)[0]
score_by_class = dict(zip(results["model"].classes_, scores))
print({"predicted_class": predicted_class, "calibrated_model_scores": score_by_class})

## ملاحظات ختامية

- النسخة الأصلية من الدفتر كانت تقرأ ملفًا نظيفًا من 478 صفًا غير مرفق؛ هذا المشروع يضم النسخة العامة ذات 480 سجلًا.
- الأداء المقاس لا يضمن التعميم على مدرسة أو بلد مختلف.
- خصائص النوع والجنسية ومكان الميلاد مستبعدة من التدريب، مع فحص وصفي محدود لفروق الأداء حسب النوع.
- استخدم النتائج لفتح نقاش دعم فردي، ولا تعتمد عليها وحدها في ترتيب الطلاب أو تقرير فرصهم.